In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib

import pickle
import time
import memory_profiler

%load_ext memory_profiler

from pathlib import Path
import distro

%load_ext watermark

In [ ]:
# %load_ext IPython.extensions.autoreload
# %autoreload 2

# from llm_readability.word_len import *

In [ ]:
import black
import jupyter_black

jupyter_black.load(line_length=79)

In [ ]:
variables_path = Path("../results/variables")
figures_path = Path("../results/figures/updated_dataset")
data_path = Path("../data")
berenslab_data_path = Path("/gpfs01/berens/data/data/pubmed_processed/")

In [ ]:
plt.style.use("matplotlib_style.txt")

In [ ]:
%watermark -a 'Rita González-Márquez' -t -d -tz -u -v -iv -w -m -h -p transformers -p openTSNE
print(distro.name(pretty=True))

Author: Rita González-Márquez

Last updated: 2025-07-25 14:17:19CEST

Python implementation: CPython
Python version       : 3.12.9
IPython version      : 9.4.0

openTSNE: not installed

Compiler    : Clang 19.1.6 
OS          : Linux
Release     : 4.18.0-553.el8_10.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 64
Architecture: 64bit

Hostname: gber7

memory_profiler: 0.61.0
pathlib        : 1.0.1
black          : 25.1.0
distro         : 1.9.0
pandas         : 2.3.1
jupyter_black  : 0.4.0
numpy          : 2.2.6
matplotlib     : 3.10.3

Watermark: 2.5.0

Rocky Linux 8.10 (Green Obsidian)


# Test word length

In [ ]:
def create_sample_corpus(size=100):
    import random

    years = list(range(2010, 2025))
    sample_texts = [
        "The quick brown fox jumps over the lazy dog",
        "Natural language processing is fascinating and revolutionary",
        "Machine learning algorithms are powerful computational tools",
        "Data science combines statistical analysis and programming",
        "Large language models transform text analysis capabilities",
        "Deep learning neural networks excel at pattern recognition",
        "Artificial intelligence research advances rapidly each year",
        "Computer vision systems process visual information effectively",
    ]
    random.seed(42)

    corpus_data = []
    for i in range(size):
        text = random.choice(sample_texts)
        year = random.choice(years)
        corpus_data.append((text, year))

    df = pd.DataFrame(
        {
            "AbstractText": [elem[0] for elem in corpus_data],
            "Year": [elem[1] for elem in corpus_data],
        }
    )
    return df

In [ ]:
test_corpus = create_sample_corpus()
test_corpus

,AbstractText,Year
0,Natural language processing is fascinating and...,2010
1,Large language models transform text analysis ...,2013
2,Data science combines statistical analysis and...,2012
3,Natural language processing is fascinating and...,2020
4,Natural language processing is fascinating and...,2019
...,...,...
95,Machine learning algorithms are powerful compu...,2014
96,Artificial intelligence research advances rapi...,2013
97,Data science combines statistical analysis and...,2021
98,Large language models transform text analysis ...,2016


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


def vectorize_abstracts(corpus_df):
    vectorizer = CountVectorizer(binary=True, min_df=1e-6)
    X = vectorizer.fit_transform(corpus_df.AbstractText.values)
    print(type(X))
    print(f"Count matrix computed: {X.shape}", flush=True)

    words = vectorizer.get_feature_names_out()
    years = np.arange(2010, 2026)
    counts = np.zeros((words.size, years.size))
    totals = np.zeros(years.size)

    for i, year in enumerate(years):
        # ind = np.array(df.Year == year)
        ind = corpus_df.Year.values == year
        print(ind)
        # X[ind, :]
        counts[:, i] = np.array(np.sum(X[ind, :], axis=0)).ravel()
        totals[i] = np.sum(ind)

    df = pd.DataFrame(
        dict(zip(["word"] + list(years), [words] + list(counts.astype(int).T)))
    )
    df.loc[len(df)] = [""] + list(totals.astype(int))

    return X, words, years, counts, totals, df

In [ ]:
X, words, years, counts, totals, df = vectorize_abstracts(test_corpus)

<class 'scipy.sparse._csr.csr_matrix'>
Count matrix computed: (100, 54)
[ True False False False False  True False False False False False False
 False False False False False False False False False False False False
 False False False False False False  True False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False]
[False False False False False False  True False False False False False
 False False False False False False  True False False False False False
 False False  True False False False False False  True False False False
 False False False False False False False False False False False False
 False False  True False F

In [ ]:
df

,word,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,advances,1,1,0,2,0,2,0,0,0,2,0,0,0,0,1,0
1,algorithms,0,0,1,0,1,0,0,1,3,0,0,1,1,0,0,0
2,analysis,0,2,6,3,0,0,2,1,2,0,2,6,5,3,2,0
3,and,2,1,5,1,0,1,3,0,0,2,4,4,3,2,2,0
4,are,0,0,1,0,1,0,0,1,3,0,0,1,1,0,0,0
5,artificial,1,1,0,2,0,2,0,0,0,2,0,0,0,0,1,0
6,at,0,2,1,1,0,1,1,2,0,2,0,0,1,1,1,0
7,brown,0,2,0,2,0,0,0,0,1,1,1,2,2,1,0,0
8,capabilities,0,2,2,2,0,0,2,1,2,0,0,2,3,2,1,0
9,combines,0,0,4,1,0,0,0,0,0,0,2,4,2,1,1,0


### debug

In [ ]:
years = np.arange(2010, 2026)

In [ ]:
# Remove the totals row (last row with empty word)
word_counts_df = df[df["word"] != ""].copy()
print(word_counts_df)

# Compute word lengths vectorized
word_lengths = word_counts_df["word"].str.len().values
print(word_lengths)

             word  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  \
0        advances     1     1     0     2     0     2     0     0     0     2   
1      algorithms     0     0     1     0     1     0     0     1     3     0   
2        analysis     0     2     6     3     0     0     2     1     2     0   
3             and     2     1     5     1     0     1     3     0     0     2   
4             are     0     0     1     0     1     0     0     1     3     0   
5      artificial     1     1     0     2     0     2     0     0     0     2   
6              at     0     2     1     1     0     1     1     2     0     2   
7           brown     0     2     0     2     0     0     0     0     1     1   
8    capabilities     0     2     2     2     0     0     2     1     2     0   
9        combines     0     0     4     1     0     0     0     0     0     0   
10  computational     0     0     1     0     1     0     0     1     3     0   
11       computer     0     

In [ ]:
year_columns

['2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025']

In [ ]:
df.columns[1]

np.int64(2010)

In [ ]:
# Extract count matrix (all year columns)
year_columns = [str(year) for year in years]
count_matrix = word_counts_df[years].values

In [ ]:
count_matrix

array([[1, 1, 0, 2, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0],
       [0, 0, 1, 0, 1, 0, 0, 1, 3, 0, 0, 1, 1, 0, 0, 0],
       [0, 2, 6, 3, 0, 0, 2, 1, 2, 0, 2, 6, 5, 3, 2, 0],
       [2, 1, 5, 1, 0, 1, 3, 0, 0, 2, 4, 4, 3, 2, 2, 0],
       [0, 0, 1, 0, 1, 0, 0, 1, 3, 0, 0, 1, 1, 0, 0, 0],
       [1, 1, 0, 2, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0],
       [0, 2, 1, 1, 0, 1, 1, 2, 0, 2, 0, 0, 1, 1, 1, 0],
       [0, 2, 0, 2, 0, 0, 0, 0, 1, 1, 1, 2, 2, 1, 0, 0],
       [0, 2, 2, 2, 0, 0, 2, 1, 2, 0, 0, 2, 3, 2, 1, 0],
       [0, 0, 4, 1, 0, 0, 0, 0, 0, 0, 2, 4, 2, 1, 1, 0],
       [0, 0, 1, 0, 1, 0, 0, 1, 3, 0, 0, 1, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 0, 2, 0, 3, 1, 0, 0, 0, 1, 0, 0],
       [0, 0, 4, 1, 0, 0, 0, 0, 0, 0, 2, 4, 2, 1, 1, 0],
       [0, 2, 1, 1, 0, 1, 1, 2, 0, 2, 0, 0, 1, 1, 1, 0],
       [0, 2, 0, 2, 0, 0, 0, 0, 1, 1, 1, 2, 2, 1, 0, 0],
       [1, 1, 0, 2, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0],
       [0, 1, 1, 0, 0, 0, 2, 0, 3, 1, 0, 0, 0, 1, 0, 0],
       [0, 2, 1, 1, 0, 1, 1, 2,

# Test tokenizer

In [ ]:
sample_texts = [
    "The quick brown fox jumps over the lazy dog",
    "The qui3ck 23546 fox-344 1jumps over-23 the lazy www.me.com dog",
]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


vectorizer = CountVectorizer(
    lowercase=True,
    # token_pattern=r"\b[a-zA-Z]+\b",  # Only alphabetic words
    token_pattern=r"\b[a-zA-Z]{4,}\b",
    # max_features=max_features,
    binary=False,  # We want counts, not binary presence
    dtype=np.int64,
)

In [ ]:
tokenizer = vectorizer.build_tokenizer()
print(tokenizer(sample_texts[0]))
print(tokenizer(sample_texts[1]))

['quick', 'brown', 'jumps', 'over', 'lazy']
['over', 'lazy']


In [ ]:
tokenizer = vectorizer.build_tokenizer()
print(tokenizer(sample_texts[0]))
print(tokenizer(sample_texts[1]))

['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
['The', 'fox', 'over', 'the', 'lazy', 'www', 'me', 'com', 'dog']
